In [ ]:
# CSV 파일을 읽어서 SQLite에 저장하는 실습

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime

In [ ]:
base_dir = Path(".")
interim_dir = base_dir / "data" / "interim"
output_dir = base_dir / "data" / "output"

In [ ]:
interim_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
input_file = interim_dir / "ai4i_enriched.csv"
df = pd.read_csv(input_file)
df.shape

In [ ]:
df.head(3)

In [ ]:
# SQLite 연결, SQLite는 파일 기반의 데이터베이스

In [ ]:
db_file = output_dir / "manu.db"
conn = sqlite3.connect(db_file)

In [ ]:
# 기존 테이블 삭제 후 적재
conn.execute("DROP TABLE IF EXISTS manufact") # 실행, manufact는 테이블명
conn.commit() # 완료

In [ ]:
df.to_sql("manufact", conn, if_exists="replace", index=False) # 1줄

In [ ]:
# 형식 : SELECT * FROM 테이블
row_count_df = pd.read_sql("SELECT COUNT(*) AS cnt FROM manufact", conn)
row_count_df

In [ ]:
db_row_count = row_count_df.loc[0, "cnt"]
db_row_count

In [ ]:
df.shape

In [ ]:
df.shape[0]

In [ ]:
# 테이블의 상위 5개 얻기
df2 = pd.read_sql("SELECT * FROM manufact LIMIT 5", conn)
df2

In [ ]:
df.head(5)

In [ ]:
# type과 평균 공구 마모 시간을 그룹으로 묶어서 검색
sql1 = """
SELECT
    type,
    ROUND( AVG( tool_wear_min ), 2 ) as avg_tool_wear
FROM
    manufact
GROUP BY type
ORDER BY type
"""
result1 = pd.read_sql(sql1, conn)
display(result1)
display( df.groupby('type')['tool_wear_min'].mean().round(2) ) # 1줄

In [ ]:
# machine_failure 컬럼(정답 y) 갯수 확인하는 SQL 실습
sql2 = """
SELECT
    machine_failure,
    COUNT(*) as cnt
FROM manufact
GROUP BY machine_failure
ORDER BY machine_failure
"""
result2 = pd.read_sql(sql2, conn)
display( result2 )
display( df['machine_failure'].value_counts() ) # 1줄
display( df.groupby('machine_failure').size().reset_index(name='cnt') ) # 1줄

In [ ]:
# failure_risk_note 컬럼을 그룹으로 갯수를 세서 역순으로 정렬
sql3 = """
SELECT
    failure_risk_note,
    COUNT(*) AS cnt
FROM manufact
GROUP BY failure_risk_note
ORDER BY cnt DESC
"""
result3 = pd.read_sql(sql3, conn)
display( result3 )
display( df['failure_risk_note'].value_counts().reset_index() ) # 1줄

In [ ]:
# DB 연결 종료
conn.close()

In [ ]:
# end